In [2]:
# Cell 1 — connect Google Drive, where the data will live
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/nbi'
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
print(os.listdir(DATA_DIR))

Mounted at /content/drive
['raw']


In [ ]:
# Cell 2 — clone the repo (or update it if already cloned) and install the libraries
import os
REPO_DIR = '/content/bridge-deterioration-nbi'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q https://github.com/AHichem/bridge-deterioration-nbi.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — open a DuckDB database stored on Drive and enable SQL cells
%load_ext sql
%sql duckdb:////content/drive/MyDrive/nbi/nbi.duckdb

In [ ]:
# Cell 3 — open a DuckDB database stored on Drive and enable SQL cells
%load_ext sql
%sql duckdb:////content/drive/MyDrive/nbi/nbi.duckdb

In [ ]:
%%sql
-- Cell 4 — a toy table to warm up
CREATE OR REPLACE TABLE toy_bridges AS
SELECT * FROM (VALUES
  ('B001', 'CA', 1965, 5),
  ('B002', 'CA', 1998, 7),
  ('B003', 'TX', 1952, 4),
  ('B004', 'TX', 2010, 8),
  ('B005', 'NY', 1931, 3),
  ('B006', 'NY', 1975, 6)
) AS t(bridge_id, state, year_built, deck_rating);

SELECT * FROM toy_bridges;

---
# SQL refresher — 1 hour on bridge data

Each section takes about 5–8 minutes and follows the same pattern:
1. **Read** the short explanation.
2. **Run** the example cell and look at the result.
3. **Do** the exercises in the empty cells (starting with `-- Your answer`).
   If you run one of these cells while it's still empty, you'll get an error (`cannot access local variable 'result'`). That's normal: write a query under the comment first.

Solutions are at the very bottom of the notebook, so try each exercise before you scroll down.

**Plan (≈ 60 min)**

| # | Topic | Min |
|---|-------|-----|
| 0 | Practice data | 3 |
| 1 | `SELECT` / `FROM` | 5 |
| 2 | `WHERE` (filtering) | 8 |
| 3 | `ORDER BY`, `LIMIT`, `DISTINCT` | 5 |
| 4 | Calculated columns, aliases, `CASE` | 8 |
| 5 | `NULL` (missing values) | 4 |
| 6 | Aggregates and `GROUP BY` | 8 |
| 7 | `HAVING` | 4 |
| 8 | `JOIN` | 8 |
| 9 | Subqueries and `WITH` (CTEs) | 5 |
| 10 | `CREATE TABLE AS` / `VIEW` | 2 |
| ★ | Bonus: window functions (`LAG`) to measure deterioration | extra |

**Order a query is written in:** `SELECT … FROM … JOIN … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT`

**Order the database actually runs it in:** `FROM/JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`.
This explains most beginner errors (for example, why you can't use a `SELECT` alias inside `WHERE`).

## 0 · Practice data (3 min)

The real NBI files arrive later. For now we create three small tables that use the same ideas as the NBI:

- **`bridges`**: one row per bridge. The condition ratings (`deck`, `superstructure`, `substructure`) use the NBI **0–9 scale**: 9 = excellent, 7 = good, 5 = fair, **4 or less = poor**. `adt` = *average daily traffic* (vehicles per day).
- **`states`**: state code → name and region.
- **`inspections`**: deck rating history over time for a few bridges (used in the bonus section).

Run the cell below once. It shows nothing at the end. That's expected.

In [ ]:
# Show up to 50 rows per result instead of the default 10
%config SqlMagic.displaylimit = 50

In [ ]:
%%sql
CREATE OR REPLACE TABLE bridges AS
SELECT * FROM (VALUES
  ('B001', 'CA', 1965, 'Concrete',             45000, 5,    6, 6, 2024),
  ('B002', 'CA', 1998, 'Steel',                82000, 7,    7, 8, 2024),
  ('B003', 'TX', 1952, 'Steel',                12000, 4,    4, 5, 2023),
  ('B004', 'TX', 2010, 'Concrete',             30500, 8,    8, 8, 2024),
  ('B005', 'NY', 1931, 'Steel',                 5600, 3,    4, 4, 2022),
  ('B006', 'NY', 1975, 'Concrete',             23000, 6,    6, 5, 2024),
  ('B007', 'FL', 1988, 'Concrete',             61000, 7,    6, 7, 2023),
  ('B008', 'FL', 1960, 'Timber',                 900, 4,    3, 4, 2021),
  ('B009', 'OH', 1949, 'Steel',                 3100, NULL, 5, 4, 2023),
  ('B010', 'OH', 2005, 'Concrete',             15800, 7,    8, 7, 2024),
  ('B011', 'CA', 1972, 'Prestressed concrete', 120000, 6,   6, 7, 2024),
  ('B012', 'TX', 1983, 'Prestressed concrete', 54000, 6,    7, 6, 2024),
  ('B013', 'NY', 2018, 'Steel',                 9800, 9,    9, 8, 2024),
  ('B014', 'FL', 1939, 'Timber',                 450, 2,    3, 3, 2020),
  ('B015', 'OH', 1968, 'Concrete',              7200, 5,    5, 5, 2022),
  ('B016', 'TX', 1995, 'Steel',                26000, 7,    6, 6, 2023),
  ('B017', 'CA', 1958, 'Concrete',              1500, 4,    5, 4, 2021),
  ('B018', 'NY', 1990, 'Prestressed concrete', 41000, 7,    7, 7, 2024),
  ('B019', 'PA', 1970, 'Steel',                18000, 5,    5, 6, 2023),
  ('B020', 'FL', 2015, 'Concrete',             33000, 8,    8, 9, 2024)
) AS t(bridge_id, state, year_built, material, adt,
       deck, superstructure, substructure, last_inspection);

CREATE OR REPLACE TABLE states AS
SELECT * FROM (VALUES
  ('CA', 'California', 'West'),
  ('TX', 'Texas',      'South'),
  ('NY', 'New York',   'Northeast'),
  ('FL', 'Florida',    'South'),
  ('OH', 'Ohio',       'Midwest'),
  ('WA', 'Washington', 'West')
) AS t(state, state_name, region);

CREATE OR REPLACE TABLE inspections AS
SELECT * FROM (VALUES
  ('B003', 2015, 6), ('B003', 2017, 6), ('B003', 2019, 5), ('B003', 2021, 5), ('B003', 2023, 4),
  ('B005', 2014, 6), ('B005', 2016, 5), ('B005', 2018, 4), ('B005', 2020, 4), ('B005', 2022, 3),
  ('B013', 2018, 9), ('B013', 2020, 9), ('B013', 2022, 9), ('B013', 2024, 9),
  ('B014', 2012, 4), ('B014', 2014, 4), ('B014', 2016, 3), ('B014', 2018, 3), ('B014', 2020, 2)
) AS t(bridge_id, inspection_year, deck);

Notice two things in the data. They are traps that show up later:
- Bridge **B009** has no deck rating (`NULL`).
- Bridge **B019** is in **PA**, which isn't in `states`, and **WA** is in `states` but has no bridges.

Use `DESCRIBE` to see a table's columns and their types:

In [ ]:
%%sql
DESCRIBE bridges;

## 1 · `SELECT` and `FROM` — pick columns (5 min)

```sql
SELECT column1, column2      -- which columns
FROM   table_name;           -- from which table
```
- `SELECT *` = all columns. It's fine for exploring, but avoid it in final queries.
- SQL keywords are not case-sensitive. Writing them in CAPITALS is just a readability convention.
- `--` starts a comment.

In [ ]:
%%sql
SELECT bridge_id, state, year_built
FROM bridges;

**Exercise 1.1**: Show every column of `states`.

**Exercise 1.2**: Show only `bridge_id`, `material` and `adt` from `bridges`.

In [ ]:
%%sql
-- Your answer: 1.1

In [ ]:
%%sql
-- Your answer: 1.2

## 2 · `WHERE` — keep only some rows (8 min)

```sql
SELECT ... FROM ... WHERE condition;
```
| Operator | Meaning | Example |
|---|---|---|
| `=`, `<>` (or `!=`) | equal / not equal | `state = 'TX'` |
| `<`, `<=`, `>`, `>=` | comparisons | `deck <= 4` |
| `AND`, `OR`, `NOT` | combine conditions | `state = 'TX' AND deck <= 4` |
| `IN (...)` | equal to any value in a list | `state IN ('CA', 'NY')` |
| `BETWEEN a AND b` | range, **ends included** | `year_built BETWEEN 1950 AND 1959` |
| `LIKE` | text pattern: `%` = any text, `_` = one character | `material LIKE '%concrete%'` |

Text goes in **single quotes** `'...'`. Text comparisons are case-sensitive (`ILIKE` ignores case).
Use parentheses when you mix `AND` and `OR`.

In [ ]:
%%sql
-- Poor-condition decks (rating 4 or less) in Texas
SELECT bridge_id, state, deck
FROM bridges
WHERE state = 'TX' AND deck <= 4;

**Exercise 2.1**: Bridges built **before 1960**.

**Exercise 2.2**: Bridges in **California or Florida** (use `IN`).

**Exercise 2.3**: Bridges built in the **1960s** (1960–1969) with more than **5,000 vehicles/day**.

**Exercise 2.4**: Bridges whose material contains the word *concrete* in any capitalisation. This should return both "Concrete" and "Prestressed concrete".

**Exercise 2.5**: Bridges in poor condition on **any** of the three components (deck, superstructure or substructure ≤ 4).

In [ ]:
%%sql
-- Your answer: 2.1

In [ ]:
%%sql
-- Your answer: 2.2

In [ ]:
%%sql
-- Your answer: 2.3

In [ ]:
%%sql
-- Your answer: 2.4

In [ ]:
%%sql
-- Your answer: 2.5

## 3 · `ORDER BY`, `LIMIT`, `DISTINCT` (5 min)

```sql
SELECT ... FROM ...
ORDER BY col1 DESC, col2 ASC   -- ASC (default) = smallest first, DESC = largest first
LIMIT 5;                       -- keep only the first 5 rows
```
`SELECT DISTINCT col` removes duplicate rows from the result.

In [ ]:
%%sql
-- The 3 busiest bridges
SELECT bridge_id, state, adt
FROM bridges
ORDER BY adt DESC
LIMIT 3;

**Exercise 3.1**: The 5 **oldest** bridges (oldest first).

**Exercise 3.2**: The list of distinct **materials**, in alphabetical order.

**Exercise 3.3**: All bridges sorted by **state** (A→Z), then by **deck rating** (worst first) within each state.

In [ ]:
%%sql
-- Your answer: 3.1

In [ ]:
%%sql
-- Your answer: 3.2

In [ ]:
%%sql
-- Your answer: 3.3

## 4 · Calculated columns, aliases (`AS`) and `CASE` (8 min)

You can calculate new columns inside `SELECT` and name them with `AS`:
```sql
SELECT bridge_id, 2024 - year_built AS age FROM bridges;
```
Useful functions: `ROUND(x, 1)`, `LEAST(a, b, c)` / `GREATEST(...)` (smallest/largest value **across columns**),
`UPPER()`, `LOWER()`, `||` to join text together.

`CASE` is SQL's *if / else*. The first condition that is true wins:
```sql
CASE
  WHEN condition1 THEN 'value1'
  WHEN condition2 THEN 'value2'
  ELSE 'other'
END AS new_column
```

In [ ]:
%%sql
-- Age of each bridge and the worst of its three ratings
SELECT bridge_id,
       2024 - year_built                         AS age,
       LEAST(deck, superstructure, substructure) AS worst_rating
FROM bridges;

**Exercise 4.1**: Show `bridge_id`, `adt` and the traffic in **thousands of vehicles**, rounded to 1 decimal, named `adt_k`.

**Exercise 4.2**: The FHWA classifies a bridge by its *lowest* rating: **Good** if it's ≥ 7, **Fair** if it's 5–6, **Poor** if it's ≤ 4.
Add a column `condition` with that label. (Hint: `CASE` on `LEAST(deck, superstructure, substructure)`.)

**Exercise 4.3**: Build a label like `B001 (CA)` by joining `bridge_id` and `state` with `||`.

In [ ]:
%%sql
-- Your answer: 4.1

In [ ]:
%%sql
-- Your answer: 4.2

In [ ]:
%%sql
-- Your answer: 4.3

## 5 · `NULL`: missing values (4 min)

`NULL` means *unknown*. It is **not** 0 and not an empty string.
- `deck = NULL` never works (it's never true). Use **`IS NULL`** / **`IS NOT NULL`**.
- Any calculation involving `NULL` gives `NULL`, and `LEAST()` / `GREATEST()` skip `NULL`s in DuckDB.
- `COALESCE(a, b)` returns `a`, or `b` if `a` is `NULL`. Use it to fill in missing values.

In the real NBI, culverts have no deck, so missing values are common.

In [ ]:
%%sql
SELECT bridge_id, deck
FROM bridges
WHERE deck IS NULL;

**Exercise 5.1**: Show `bridge_id` and `deck`, but display **-1** when the deck rating is missing.

**Exercise 5.2**: Run the two queries below and explain why the counts differ: `SELECT COUNT(*) FROM bridges WHERE deck <= 4;` and
`SELECT COUNT(*) FROM bridges WHERE NOT (deck <= 4);`. Their total isn't 20. Which bridge is missing from both?

In [ ]:
%%sql
-- Your answer: 5.1

In [ ]:
%%sql
-- Your answer: 5.2

## 6 · Aggregates and `GROUP BY` (8 min)

Aggregate functions combine many rows into **one value**:
`COUNT(*)` (rows), `COUNT(col)` (values that aren't `NULL`), `SUM`, `AVG`, `MIN`, `MAX`.

`GROUP BY` splits the rows into groups and calculates the aggregates **once per group**:
```sql
SELECT state, COUNT(*) AS n_bridges, AVG(deck) AS avg_deck
FROM bridges
GROUP BY state;
```
**Rule:** every column in `SELECT` must either be in `GROUP BY` or be inside an aggregate function.

DuckDB tip: `GROUP BY ALL` automatically groups by every column that isn't aggregated.

A handy trick: `AVG(CASE WHEN deck <= 4 THEN 1 ELSE 0 END)` = the **share** of rows meeting a condition.

In [ ]:
%%sql
-- Overall summary of the whole table: one row
SELECT COUNT(*)              AS n_bridges,
       COUNT(deck)           AS n_with_deck_rating,
       ROUND(AVG(deck), 2)   AS avg_deck,
       MIN(year_built)       AS oldest,
       SUM(adt)              AS total_daily_traffic
FROM bridges;

**Exercise 6.1**: Number of bridges and average deck rating (rounded to 2 decimals) **per state**, sorted by average deck rating, worst first.

**Exercise 6.2**: Per **material**: number of bridges, average age in 2024, and the share (%) of bridges with a poor deck (≤ 4).

**Exercise 6.3**: Number of bridges per **decade built** (1930, 1940, …). Hint: `(year_built // 10) * 10` (`//` = integer division).

In [ ]:
%%sql
-- Your answer: 6.1

In [ ]:
%%sql
-- Your answer: 6.2

In [ ]:
%%sql
-- Your answer: 6.3

## 7 · `HAVING`: filter groups (4 min)

`WHERE` filters **rows before** grouping. `HAVING` filters **groups after** grouping, so it can use aggregates.
```sql
SELECT state, COUNT(*) AS n
FROM bridges
WHERE year_built < 2000      -- rows
GROUP BY state
HAVING COUNT(*) >= 3;        -- groups
```

In [ ]:
%%sql
SELECT state, COUNT(*) AS n
FROM bridges
WHERE year_built < 2000
GROUP BY state
HAVING COUNT(*) >= 3;

**Exercise 7.1**: States whose **average deck rating is below 6**.

**Exercise 7.2**: Materials with **at least 2 bridges** in poor deck condition (≤ 4). Hint: combine `WHERE` and `HAVING`.

In [ ]:
%%sql
-- Your answer: 7.1

In [ ]:
%%sql
-- Your answer: 7.2

## 8 · `JOIN`: combine tables (8 min)

```sql
SELECT b.bridge_id, s.state_name
FROM bridges AS b                   -- b and s are short table aliases
JOIN states  AS s ON b.state = s.state;   -- matching rule
```
| Type | Keeps |
|---|---|
| `JOIN` (= `INNER JOIN`) | only rows that match in **both** tables |
| `LEFT JOIN` | **every** row of the left table, with `NULL`s where there's no match |

Remember the traps: **PA** (a bridge with no state row) and **WA** (a state with no bridges).

In [ ]:
%%sql
SELECT b.bridge_id, b.state, s.state_name, s.region
FROM bridges AS b
JOIN states  AS s ON b.state = s.state;

**Exercise 8.1**: Run the query above. How many rows does it return, and which bridge is missing? Rewrite it so that bridge appears (with a `NULL` state name).

**Exercise 8.2**: Number of bridges and average deck rating **per region**.

**Exercise 8.3**: List **every state** in `states` with its number of bridges, so that **Washington shows 0**.
Hint: start `FROM states` and `LEFT JOIN bridges`, then use `COUNT(b.bridge_id)`, not `COUNT(*)`. Why?

In [ ]:
%%sql
-- Your answer: 8.1

In [ ]:
%%sql
-- Your answer: 8.2

In [ ]:
%%sql
-- Your answer: 8.3

## 9 · Subqueries and `WITH` (CTEs) (5 min)

A **subquery** is a query inside another query:
```sql
SELECT bridge_id, deck FROM bridges
WHERE deck < (SELECT AVG(deck) FROM bridges);   -- compare to the overall average
```
A **CTE** (`WITH name AS (...)`) gives a subquery a name. It reads top-to-bottom like a recipe and is the
preferred style for anything with more than one step:
```sql
WITH step1 AS ( ... ),
     step2 AS ( SELECT ... FROM step1 ... )
SELECT ... FROM step2;
```

In [ ]:
%%sql
WITH rated AS (
    SELECT bridge_id, state,
           LEAST(deck, superstructure, substructure) AS worst_rating
    FROM bridges
)
SELECT state, COUNT(*) AS n_poor
FROM rated
WHERE worst_rating <= 4
GROUP BY state
ORDER BY n_poor DESC;

**Exercise 9.1**: Bridges whose traffic (`adt`) is **above the average traffic** of all bridges (use a subquery).

**Exercise 9.2**: Using a CTE, calculate the average deck rating per state, then keep only states **above the overall average** deck rating.

In [ ]:
%%sql
-- Your answer: 9.1

In [ ]:
%%sql
-- Your answer: 9.2

## 10 · Save results: `CREATE TABLE AS` and `VIEW` (2 min)

```sql
CREATE OR REPLACE TABLE name AS SELECT ...;   -- stores a copy of the result (fast to reuse)
CREATE OR REPLACE VIEW  name AS SELECT ...;   -- stores the query only (always up to date)
```
Because the database lives in `nbi.duckdb` on your Drive, tables you create **persist between Colab sessions**.

You will use this all the time in the project: raw NBI → cleaned table → analysis tables.

**Exercise 10.1**: Create a view `bridge_condition` with `bridge_id`, `state`, `age` (in 2024) and the `condition` label from exercise 4.2.
Then query the view to count bridges per `condition`.

In [ ]:
%%sql
-- Your answer: 10.1

## ★ Bonus: window functions to measure deterioration

A **window function** calculates a value across related rows **without collapsing them** (unlike `GROUP BY`).
```sql
LAG(deck) OVER (PARTITION BY bridge_id ORDER BY inspection_year)
```
returns the deck rating from the **previous inspection of the same bridge**. That's exactly what you need to measure
how fast bridges lose condition points, which is the core of this project.

Other useful ones: `ROW_NUMBER()`, `RANK()`, `AVG(...) OVER (PARTITION BY ...)`.

In [ ]:
%%sql
SELECT bridge_id, inspection_year, deck,
       LAG(deck) OVER (PARTITION BY bridge_id ORDER BY inspection_year)        AS prev_deck,
       deck - LAG(deck) OVER (PARTITION BY bridge_id ORDER BY inspection_year) AS change
FROM inspections
ORDER BY bridge_id, inspection_year;

**Exercise ★.1**: For each bridge in `inspections`, calculate the **average deterioration rate in rating points per year**:
(last rating − first rating) / (last year − first year). Which bridge deteriorates fastest?
Hint: `GROUP BY bridge_id` with `arg_min(deck, inspection_year)` / `arg_max(deck, inspection_year)` (DuckDB functions: the `deck` value at the min/max year).

**Exercise ★.2**: For each state, show only its **busiest bridge** (highest `adt`). Hint: `ROW_NUMBER() OVER (PARTITION BY state ORDER BY adt DESC)` in a CTE, then keep row number 1.

In [ ]:
%%sql
-- Your answer: ★.1

In [ ]:
%%sql
-- Your answer: ★.2

---
# Solutions

Try the exercises first! Each cell below runs as-is. There's often more than one correct answer.

In [ ]:
%%sql
-- Solution 1.1
SELECT * FROM states;

In [ ]:
%%sql
-- Solution 1.2
SELECT bridge_id, material, adt FROM bridges;

In [ ]:
%%sql
-- Solution 2.1
SELECT * FROM bridges WHERE year_built < 1960;

In [ ]:
%%sql
-- Solution 2.2
SELECT * FROM bridges WHERE state IN ('CA', 'FL');

In [ ]:
%%sql
-- Solution 2.3
SELECT * FROM bridges
WHERE year_built BETWEEN 1960 AND 1969
  AND adt > 5000;

In [ ]:
%%sql
-- Solution 2.4
SELECT * FROM bridges WHERE material ILIKE '%concrete%';

In [ ]:
%%sql
-- Solution 2.5
SELECT * FROM bridges
WHERE deck <= 4 OR superstructure <= 4 OR substructure <= 4;

In [ ]:
%%sql
-- Solution 3.1
SELECT bridge_id, year_built FROM bridges
ORDER BY year_built
LIMIT 5;

In [ ]:
%%sql
-- Solution 3.2
SELECT DISTINCT material FROM bridges ORDER BY material;

In [ ]:
%%sql
-- Solution 3.3
SELECT bridge_id, state, deck FROM bridges
ORDER BY state ASC, deck ASC;

In [ ]:
%%sql
-- Solution 4.1
SELECT bridge_id, adt, ROUND(adt / 1000, 1) AS adt_k FROM bridges;

In [ ]:
%%sql
-- Solution 4.2
SELECT bridge_id,
       LEAST(deck, superstructure, substructure) AS worst_rating,
       CASE
         WHEN LEAST(deck, superstructure, substructure) >= 7 THEN 'Good'
         WHEN LEAST(deck, superstructure, substructure) >= 5 THEN 'Fair'
         ELSE 'Poor'
       END AS condition
FROM bridges;

In [ ]:
%%sql
-- Solution 4.3
SELECT bridge_id || ' (' || state || ')' AS label FROM bridges;

In [ ]:
%%sql
-- Solution 5.1
SELECT bridge_id, COALESCE(deck, -1) AS deck FROM bridges;

In [ ]:
%%sql
-- Solution 5.2
-- 5 + 14 = 19, not 20: for B009, deck is NULL, so `NULL <= 4` is NULL (unknown),
-- and NOT NULL is still NULL. WHERE only keeps rows where the condition is TRUE.
SELECT COUNT(*) FROM bridges WHERE deck <= 4;

In [ ]:
%%sql
-- Solution 6.1
SELECT state, COUNT(*) AS n_bridges, ROUND(AVG(deck), 2) AS avg_deck
FROM bridges
GROUP BY state
ORDER BY avg_deck;

In [ ]:
%%sql
-- Solution 6.2
SELECT material,
       COUNT(*)                                               AS n_bridges,
       ROUND(AVG(2024 - year_built), 1)                       AS avg_age,
       ROUND(100 * AVG(CASE WHEN deck <= 4 THEN 1 ELSE 0 END), 1) AS pct_poor_deck
FROM bridges
GROUP BY material
ORDER BY pct_poor_deck DESC;

In [ ]:
%%sql
-- Solution 6.3
SELECT (year_built // 10) * 10 AS decade, COUNT(*) AS n
FROM bridges
GROUP BY decade
ORDER BY decade;

In [ ]:
%%sql
-- Solution 7.1
SELECT state, ROUND(AVG(deck), 2) AS avg_deck
FROM bridges
GROUP BY state
HAVING AVG(deck) < 6;

In [ ]:
%%sql
-- Solution 7.2
SELECT material, COUNT(*) AS n_poor
FROM bridges
WHERE deck <= 4
GROUP BY material
HAVING COUNT(*) >= 2;

In [ ]:
%%sql
-- Solution 8.1
-- The inner JOIN returns 19 rows: B019 (PA) has no match in states.
SELECT b.bridge_id, b.state, s.state_name, s.region
FROM bridges AS b
LEFT JOIN states AS s ON b.state = s.state;

In [ ]:
%%sql
-- Solution 8.2
SELECT s.region, COUNT(*) AS n_bridges, ROUND(AVG(b.deck), 2) AS avg_deck
FROM bridges AS b
JOIN states AS s ON b.state = s.state
GROUP BY s.region
ORDER BY avg_deck;

In [ ]:
%%sql
-- Solution 8.3
-- COUNT(*) would count WA's single row of NULLs as 1, while COUNT(b.bridge_id) skips NULLs and gives 0.
SELECT s.state_name, COUNT(b.bridge_id) AS n_bridges
FROM states AS s
LEFT JOIN bridges AS b ON b.state = s.state
GROUP BY s.state_name
ORDER BY n_bridges DESC;

In [ ]:
%%sql
-- Solution 9.1
SELECT bridge_id, adt
FROM bridges
WHERE adt > (SELECT AVG(adt) FROM bridges)
ORDER BY adt DESC;

In [ ]:
%%sql
-- Solution 9.2
WITH per_state AS (
    SELECT state, AVG(deck) AS avg_deck
    FROM bridges
    GROUP BY state
)
SELECT state, ROUND(avg_deck, 2) AS avg_deck
FROM per_state
WHERE avg_deck > (SELECT AVG(deck) FROM bridges);

In [ ]:
%%sql
-- Solution 10.1
CREATE OR REPLACE VIEW bridge_condition AS
SELECT bridge_id, state,
       2024 - year_built AS age,
       CASE
         WHEN LEAST(deck, superstructure, substructure) >= 7 THEN 'Good'
         WHEN LEAST(deck, superstructure, substructure) >= 5 THEN 'Fair'
         ELSE 'Poor'
       END AS condition
FROM bridges;

SELECT condition, COUNT(*) AS n
FROM bridge_condition
GROUP BY condition;

In [ ]:
%%sql
-- Solution ★.1
SELECT bridge_id,
       MIN(inspection_year) AS first_year,
       MAX(inspection_year) AS last_year,
       ROUND((arg_max(deck, inspection_year) - arg_min(deck, inspection_year))
             / (MAX(inspection_year) - MIN(inspection_year)), 3) AS points_per_year
FROM inspections
GROUP BY bridge_id
ORDER BY points_per_year;   -- most negative = fastest deterioration (B005)

In [ ]:
%%sql
-- Solution ★.2
WITH ranked AS (
    SELECT bridge_id, state, adt,
           ROW_NUMBER() OVER (PARTITION BY state ORDER BY adt DESC) AS rn
    FROM bridges
)
SELECT bridge_id, state, adt
FROM ranked
WHERE rn = 1
ORDER BY adt DESC;